In [ ]:
import numpy as np
import pandas as pd

In [ ]:
telework_total = {
    "Belgium": [67324.38448, 438471.9682, 796806.7088, 2738689.039],
    "BCR": [10035.74586, 69133.2125, 97633.21467, 243345.2156],
}

FREQ_DAYS = [5.0, 3.0, 1.5, 0.0]

In [ ]:
def parse_age_bracket(age_str):
    """Extract lower bound from age bracket string like '25-29', '100+'."""
    if pd.isna(age_str):
        return 0
    age_str = str(age_str).strip()
    if '+' in age_str:
        return int(age_str.replace('+', ''))
    return int(age_str.split('-')[0])

def map_education(education_code):
    education_mapping = {
        'ISCED11_1': 'Primary',
        'ISCED11_2': 'Secondary',
        'ISCED11_3': 'Secondary',
        'ISCED11_4': 'Secondary',
        'ISCED11_5': 'Tertiary',
        'ISCED11_6': 'Tertiary',
        'ISCED11_7': 'Tertiary',
        'ISCED11_8': 'Tertiary',
        'UNK': 'Total',
        'NONE': 'Total',
    }

    return education_mapping.get(education_code, 'Total')

def compute_p_today(counts):
    """
    Convert [Jamais, Parfois, Habituellement, Toujours] counts
    to p(telework on a given day).
    p_today = sum(n_i * days_i/5) / sum(n_i)
    Returns 0 if total is 0 (unreliable cell).
    """
    total = sum(counts)
    if total == 0:
        return None
    return sum(c * d / 5.0 for c, d in zip(counts, FREQ_DAYS)) / total

In [ ]:
def validate_telework(pop):
    print("=== Global telework stats ===")
    rate = pop['telework_today'].mean()
    n_commuters = (pop['telework_today'] == 0).sum()
    print(f"Share teleworking today:      {rate:.3f}")
    print(f"Commuters remaining:          {n_commuters:,} ({1-rate:.1%})")
    print(f"Overall BCR probability of telework today: {compute_p_today(telework_total["BCR"]):.3f}")
    print()

    print("=== Telework rate by industry (model output) ===")
    print(
        pop.groupby('industry')['telework_today']
        .mean().round(3).sort_values(ascending=False)
    )
    print()

    print("=== Telework rate by sex ===")
    print(pop.groupby('sex')['telework_today'].mean().round(3))
    print()

    print("=== Telework rate by professional status ===")
    print(pop.groupby('professional_status')['telework_today'].mean().round(3))
    print()

    print("=== Telework rate by education ===")
    pop = pop.copy()
    pop['education_group'] = pop['education'].apply(map_education)
    print(pop.groupby('education_group', observed=True)['telework_today'].mean().round(3))
    print()

    print("=== Telework rate by age group ===")
    pop = pop.copy()
    pop['age_lower'] = pop['age'].apply(parse_age_bracket)
    bins   = [15, 25, 35, 45, 55, 65, 200]
    labels = ['15-24', '25-34', '35-44', '45-54', '55-64', '65+']
    pop['age_group'] = pd.cut(
        pop['age_lower'], bins=bins, labels=labels, right=False
    )
    print(pop.groupby('age_group', observed=True)['telework_today'].mean().round(3))
    print()

In [ ]:
# Validate results
pop_validate = pd.read_csv('output/workers_with_telework.csv')
validate_telework(pop_validate)